In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder,OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_selection import SelectKBest,RFECV,,f_regression
from sklearn.ensemble importRandomForestRegressor
from sklearn.linear_model import  LinearRegression
from sklearn.metrics import r2_score,mean_squared_error
sns.set_theme(style="darkgrid")
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
data = pd.read_csv("/kaggle/input/datasets/jigarkhadka/energy/Energy_consumption.csv")

# Feature engineering from Timestamp
data["Timestamp"] = pd.to_datetime(data["Timestamp"])
data["Month"] = data["Timestamp"].dt.month_name()
data["IsWeekend"] = data["DayOfWeek"].apply(lambda x: 1 if x in ["Sunday", "Saturday"] else 0)

# Define X without Timestamp and target column, keeping data intact
X = data.drop(columns=["EnergyConsumption", "Timestamp"])
y = data["EnergyConsumption"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## **Helper Functions**

In [ ]:
# Function to optimize 'k'
def select_k(X,y,estimator,scoring,func,k_range=range(1,5)):
  global preprocessor
  optimize = Pipeline(steps=[("preprocessor", preprocessor),
                                ("selector",SelectKBest(func)),
                                ("regressor", estimator)])

  param_grid = {'selector__k' : k_range}
  grid = GridSearchCV(optimize, param_grid=param_grid, cv=5, scoring=scoring)
  grid.fit(X, y)
  return grid.best_params_.get("selector__k")

# Function to compare 'models'
models_table = pd.DataFrame(columns=["Model","Type","R2","RMSE"])
def select_model(estimator,X,y,features,type="Baseline"):
  global models_table
  global X_test
  global y_test
  global preprocessor
  X_processed = preprocessor.fit_transform(X)
  X_test_processed = preprocessor.transform(X_test)
  model = estimator
  model.fit(X_processed[features],y)
  y_pred = model.predict(X_test_processed[features])
  r2 = r2_score(y_test,y_pred)
  rmse = mean_squared_error(y_test,y_pred)**0.5
  return [str(estimator).split("(")[0],
          type,
          r2,
          rmse]

# **Exploratory Data Analysis(EDA)**

#### Understanding the data

In [ ]:
# Checking datatypes and null values
data.info()

In [ ]:
#Checking a subset of data
data.head(10)

In [ ]:
# Checking unique values in data
data.nunique().sort_values(ascending=False)

In [ ]:
# Statistical Summary
data.describe() #only numerical values

#### Data Quality Check

In [ ]:
# Checking for Null or Duplicate values
print(data.isnull().sum().rename('Null Count').sort_values(ascending=False))
duplicates = data.duplicated().sum()
print("Number of duplicate rows", duplicates)

In [ ]:
# Separating Categorical and Numerical data
categorical_data = data.select_dtypes(include=['object'])
numerical_data = data.select_dtypes(exclude=['object','datetime64[ns]'])
print("Number of categorical columns", categorical_data.columns.size)

print(('\n').join(categorical_data.columns.to_list()),"\n")
print("Number of numerical columns", numerical_data.columns.size)
print(('\n').join(numerical_data.columns.to_list()))

## Breakdown
In the analysis so far, we've discovered that the data consists of no null values or any duplicate values. This will be helpful in training the model because the data quality is good.

We have 5 categorical columns and 6 numerical columns, with our target being a numerical column. Hence, we will be comparing two models: Linear Regression and Random Forest, to see which one makes better predictions.

However, before doing that we must check the data for any more patterns or useful insights.

## **Visualizing the data**

#### Categorical Data


In [ ]:
# Plotting categorical
figure, ax = plt.subplots(3,2,figsize=(15, 15))
i = 0
j = 0
for column in categorical_data.columns:
   sns.countplot(data=data[column], ax=ax[j,i])
   ax[j,i].set_xlabel(column,fontsize=14,weight="bold",color="red")
   ax[j,i].set_ylabel("Count",fontsize=14)
   if i < 1:
    i += 1
   elif i == 1:
    i = 0
    j += 1
   elif j == 3:
    break

figure.suptitle("Counts of Categorical Data", fontsize=20,weight="bold")
plt.tight_layout()
plt.show()

**Time-Series Plot(Energy Consumption)**

In [ ]:
sns.lineplot(data=data,x=data["Timestamp"],y=data["EnergyConsumption"])
plt.title('Time-Series Plot(Energy Consumption)',fontsize=15,weight="bold")
plt.xlabel('Timestamp')
plt.ylabel('Energy Consumption')

**Energy Consumption(HVAC and Lighting)**

In [ ]:
# Energy Consumption when HVAC is off
print("Energy Consumption HVAC -- OFF")
print(data.loc[data["HVACUsage"]=="Off", "EnergyConsumption"].describe(), "\n")

# Energy Consumption when HVAC is on
print("Energy Consumption HVAC -- ON")
print(data.loc[data["HVACUsage"]=="On", "EnergyConsumption"].describe(), "\n")

hvacoff = data.loc[data["HVACUsage"]=="Off", "EnergyConsumption"].mean().round(2)
hvacon = data.loc[data["HVACUsage"]=="On", "EnergyConsumption"].mean().round(2)

In [ ]:
# Energy Consumption when Lighting is off
print("Energy Consumption Lighting -- OFF")
print(data.loc[data["LightingUsage"]=="Off", "EnergyConsumption"].describe(), "\n")

# Energy Consumption when Lighting is on
print("Energy Consumption Lighting -- ON")
print(data.loc[data["LightingUsage"]=="On", "EnergyConsumption"].describe(), "\n")

lightoff = data.loc[data["LightingUsage"]=="Off", "EnergyConsumption"].mean().round(2)
lighton = data.loc[data["LightingUsage"]=="On", "EnergyConsumption"].mean().round(2)

In [ ]:
# Energy Consumption when Lighting and HVAC is off
print("Energy Consumption when both Lighting and HVAC -- OFF")
print(data.loc[(data["LightingUsage"]=="Off") & (data["HVACUsage"]=="Off"), "EnergyConsumption"].describe(), "\n")

# Energy Consumption when Lighting and HVAC is on
print("Energy Consumption when both Lighting and HVAC -- ON")
print(data.loc[(data["LightingUsage"]=="On") & (data["HVACUsage"]=="On"), "EnergyConsumption"].describe(), "\n")

bothon = data.loc[(data["LightingUsage"]=="On") & (data["HVACUsage"]=="On"), "EnergyConsumption"].mean().round(2)
bothoff = data.loc[(data["LightingUsage"]=="Off") & (data["HVACUsage"]=="Off"), "EnergyConsumption"].mean().round(2)

### Plotting above information

In [ ]:
# HVAC
figure, ax = plt.subplots(1,2,figsize=(10,4))
sns.barplot(x=["OFF","ON"], y=[hvacoff, hvacon],ax=ax[0], palette="viridis",hue=[hvacoff,hvacon])
ax[0].set_title("HVAC Usage",weight="bold")
ax[0].set_ylabel("Energy Consumption")

#Lighting
sns.barplot(x=["OFF","ON"], y=[lightoff, lighton],ax=ax[1], palette="rocket",hue=[lightoff,lighton])
ax[1].set_title("Lightning Usage",weight="bold")
ax[1].set_ylabel("Energy Consumption")

In [ ]:
#Both Lighting and HVAC Usage
sns.barplot(x=["OFF","ON"], y=[bothoff, bothon], palette="vlag",hue=[bothoff,bothon])
plt.title("Lightning and HVAC Usage",weight="bold")
plt.ylabel("Energy Consumption")

**Energy Consumption Based on Month**

In [ ]:
#Creating a new column ["Month"] that has the month name
months = data["Timestamp"].dt.month_name()
data["Month"] = months
data["Month"].unique()

In [ ]:
# Energy Consumption January Vs. February
jan_energy = data.loc[data["Month"]=="January", "EnergyConsumption"].mean().round(2)
feb_energy = data.loc[data["Month"]=="February", "EnergyConsumption"].mean().round(2)

# Plotting
plt.figure(figsize=(7,7))
sns.barplot(x=["January","February"], y=[jan_energy, feb_energy], palette="rocket",hue=[jan_energy, feb_energy])
plt.title("Energy Consumption Based on Month",weight="bold")
plt.ylabel("Energy Consumption")
plt.yticks(range(0,80,3))
plt.show()

### **Breakdown**

Surprisingly all of our data was collected in the month of January and February. Months do not have a huge difference in terms of energy consumption. The exact relationship will be tested below.

HVACUsage and LightingUsage both have an effect on EnergyConsumption, as intended.


# **Feature Engineering**


##### Feature Transformation
Although we already added a feature ['Month'] earlier. Let's check for any more.
Since we figured out that combing both HVAC and Lighting Usage has a effect on EnergyConsumption.<br> We will create a new feature called HLUsage that will be **'ON'** if BOTH are **'ON'** and **'OFF'** if ONE is **'OFF'**

In [ ]:
# Adding a feature called ['HLUsage']
# Combines both HVACUsage and LightingUsage
X_train['HLUsage'] = ((X_train["HVACUsage"] == "On") & (X_train["LightingUsage"] == "On")).astype(int)
X_test['HLUsage'] = ((X_test["HVACUsage"] == "On") & (X_test["LightingUsage"] == "On")).astype(int)

## Preprocessing
##### Feature Encoding

In [ ]:

# Building a pipeline to encode features + scaling
# Ordinal --> Ordinal Encoder, Nominal --> OneHotEncoder
nominal = ["DayOfWeek","Month"]
ordinal = [col for col in categorical_data.columns if col not in nominal]
numerical_columns_to_scale = numerical_data.columns.drop("EnergyConsumption").to_list()
preprocessor = ColumnTransformer(transformers=[("ordinal", OrdinalEncoder(), ordinal),
                                               ("nominal", OneHotEncoder(sparse_output=False,handle_unknown="ignore"), nominal),
                                               ("scaler", MinMaxScaler(), numerical_columns_to_scale)],
                                 remainder="passthrough")
preprocessor.set_output(transform="pandas")
preprocessor.fit(X_train)

# **Feature Selection**

In [ ]:
# Checking correlation with target variable
correlation = preprocessor.transform(X_train).corrwith(y_train)
correlation = correlation.abs().sort_values(ascending=False)
print(correlation)
sns.barplot(x=correlation.values,y=correlation.index)
plt.title("Correlation With EnergyConsumption", weight="bold")
plt.xlabel("Correlation Coefficient", color="red")
plt.ylabel("Features", color="green")
plt.show()

HVACUsage, Temperature and Occupancy seem to be the biggest factor determining EnergyConsumption which is self explanatory.

HVACUsage means that energy is being consumed.
Temperature also affects Energy consumption because during hotter weathers people tend to use fans, AC and heaters on colder weather.

Higher Occupancy means more number of people means more energy consumption.

In [ ]:
# Correlation matrix
# Ordinal Encoding the data(OneHot Encoding explodes the data making it hard to read)
corr_data = X_train.copy()
corr_data_categories = ['HVACUsage', 'LightingUsage', 'DayOfWeek', 'Holiday','Month']
encoder = OrdinalEncoder()
corr_data[corr_data_categories] = encoder.fit_transform(corr_data[corr_data_categories])

corr_matrix = corr_data.corr()
plt.figure(figsize=(10,5))
sns.heatmap(data=corr_matrix,annot=True)
plt.title("Correlation Heatmap", weight="bold")
plt.show()

**F-Regression Test**

In [ ]:
# Using F-Test Regression for Feature Selection
# Optimizing 'k' for SelectKBest in LinearRegression
k_linear = select_k(func=f_regression,scoring='r2',X=X_train,y=y_train,k_range=range(1,10),estimator=LinearRegression())

#Optimizing 'k' for RandomForestRegressor
k_random = select_k(func=f_regression,scoring='r2',X=X_train,y=y_train,k_range=range(1,10),estimator=RandomForestRegressor())

In [ ]:
 # Using F-Test To Find Optimal Features
X_for_f_test = preprocessor.transform(X_train)
linear_selector = SelectKBest(f_regression, k=k_linear)
random_selector = SelectKBest(f_regression, k=k_random)
linear_selector.fit_transform(X_for_f_test, y_train)
random_selector.fit_transform(X_for_f_test,y_train)
f_test_linear = X_for_f_test.columns[linear_selector.get_support()].to_list()
f_test_random = X_for_f_test.columns[random_selector.get_support()].to_list()
print(f_test_linear)
print(f_test_random)

In [ ]:
# Using RFECV for feature selection
rfecv_linear = RFECV(estimator=LinearRegression(), step=1, cv=5, scoring='r2')
rfecv_linear.fit(X_for_f_test, y_train)
rfecv_random = RFECV(estimator=RandomForestRegressor(random_state=42),step=1,cv=5,scoring='r2')
rfecv_random.fit(X_for_f_test,y_train)

In [ ]:
# Printing the features selected by RFECV
rfecv_linear_features = X_for_f_test.columns[rfecv_linear.get_support()].to_list()
rfecv_random_features = X_for_f_test.columns[rfecv_random.get_support()].to_list()
print("Linear Regression")
print(f"Optimal Number of Features:- {rfecv_linear.n_features_}")
print(f"Features:-\n {rfecv_linear_features}\n")
print("Random Forest Regressor")
print(f"Optimal Number of Features:- {rfecv_random.n_features_}")
print(f"Features:-\n {rfecv_random_features}")

### **Creating Baseline Models**

In [ ]:
#Defining ALL Features for baseline models
all_features = preprocessor.fit_transform(X_train).columns.to_list()
# Baseline Linear Regression
baseline_lreg = select_model(estimator=LinearRegression(),X=X_train,y=y_train,features=all_features)
models_table.loc[len(models_table)] = baseline_lreg

#Baseline RandomForestRegressor
baseline_rforest = select_model(estimator=RandomForestRegressor(random_state=42),X=X_train,y=y_train,features=all_features)
models_table.loc[len(models_table)] = baseline_rforest


### **Models With Selected Features**




In [ ]:
f# F-Test Linear Regression
f_test_lreg = select_model(estimator=LinearRegression(),X=X_train,y=y_train,features=f_test_linear,type="F-Test")
models_table.loc[len(models_table)] = f_test_lreg

# F-Test RandomForest
f_test_rforest = select_model(estimator=RandomForestRegressor(random_state=42),X=X_train,y=y_train,features=f_test_random,type="F-Test")
models_table.loc[len(models_table)] = f_test_rforest

# RFECV Linear Regression
rfecv_test_lreg = select_model(estimator=LinearRegression(),X=X_train,y=y_train,features=rfecv_linear_features,type="RFECV")
models_table.loc[len(models_table)] = rfecv_test_lreg

rfecv_test_rforest = select_model(estimator=RandomForestRegressor(random_state=42),X=X_train,y=y_train,features=rfecv_random_features,type="RFECV")
models_table.loc[len(models_table)] = rfecv_test_rforest



In [ ]:
# Viewing the models table to check the best model
models_table

#### Best Model --
LinearRegression with features selected by fregression.

## **Hyperparameter Tuning**
#### Linear Regression has no hyperparameters to tune, therefore the baseline model that uses the features selected by fregression() is the best model.

## **Final Model**

In [ ]:
final_model = Pipeline(steps=[("preprocessor", preprocessor),
                              ("selector", SelectKBest(f_regression, k=k_linear)),
                              ("model", LinearRegression())])

final_model.fit(X_train,y_train)
y_pred = final_model.predict(X_test)

print("--Linear Regression--")
print("**Metrics**")
print(f"R2 Score: {r2_score(y_test,y_pred)}")
print(f"--> The model explains {(r2_score(y_test,y_pred) * 100):.2f}% of the relationship between the features and target variable.\n")
print(f"RMSE: {mean_squared_error(y_test,y_pred)**0.5}")
print(f"--> The model is off by approx. {mean_squared_error(y_test,y_pred)**0.5:.2f} units on average")